## fMRI time series to covariance matrices


This notebook shows how raw fMRI time series are preprocessed into Gaussian-copula covariance matrices -- the core data representation used throughout the analysis.

Use the cleaned, state-grouped MATLAB inputs used by the diagnostics (configure `MAT_PATHS` below). The uncleaned Zenodo array is not interchangeable: its scan counts/order do not match `data/scan_metadata.csv`.
- **MA dataset** (Uhrig et al. 2018): https://zenodo.org/records/10572216 -- file `CoCoMac/timeseries.npy`
- **DBS dataset** (Tasserie et al. 2022): contact corresponding authors

Each loaded state array has shape `(N_scans, T_samples, 82_regions)`. Conventional regional GSR (OLS on an intercept and the mean z-scored regional signal) is fitted to each full scan in both datasets, before duration matching and the Gaussian copula. Retain the first 250 MA samples (~600 s) and every other DBS sample (250 points, ~625 s).

The existing 0.0025-0.05 Hz band-pass and MA-only notch are unchanged. No-GSR and added-DBS-notch sensitivities remain in `diagnostics/01_B_omega_sensitivity.ipynb`.

The output is `data/covariance_matrices_gc.h5`, a nested HDF5 file with one group per dataset and one dataset per brain state.


Note: This notebook should be run from the `high-order-anesthesia` folder to ensure the correct imports and file paths are used.


In [11]:
from pathlib import Path
import os
def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(
        f"Could not find '{target_name}' in current path or parents. "
        f"Please run the notebook from inside the project."
    )
ROOT = ensure_project_root("high-order-anesthesia")
print(f"Now in: {ROOT.name}")


Now in: high-order-anesthesia


In [12]:
import numpy as np
import pandas as pd
from src.hoi_anesthesia import diagnostics as dg
from src.hoi_anesthesia.preprocessing import generate_covmats # to generate dict
from src.hoi_anesthesia.io import load_covariance_dict # to load dict


In [ ]:
data_path = "data"
MAT_PATHS = {
    "MA": Path(r"C:\CAMILO\Brain_Entropy\DATA\monkeys\data_ts_Anesthesia_Cleaned.mat"),
    "DBS": Path(r"C:\CAMILO\Brain_Entropy\DATA\monkeys\data_ts_DBS_CoCoMac_0.0025-0.05_Cleaned_Camilo_version.mat"),
}
metadata = pd.read_csv(Path(data_path) / "scan_metadata.csv")
all_states = {}
for dataset, path in MAT_PATHS.items():
    states = dg.load_timeseries(path, dataset)
    dataset_metadata = metadata[metadata["dataset"] == dataset]
    if set(states) != set(dataset_metadata["state"]):
        raise ValueError(f"{dataset}: states do not match scan metadata")
    all_states[dataset] = {}
    for state, scans in states.items():
        indices = dataset_metadata.loc[dataset_metadata["state"] == state, "scan_index"]
        if not np.array_equal(np.sort(indices), np.arange(len(scans))):
            raise ValueError(f"{dataset}/{state}: scan counts/indices do not match metadata")
        if scans.ndim != 3 or scans.shape[1:] != (500, 82) or not np.isfinite(scans).all():
            raise ValueError(f"{dataset}/{state}: expected finite (n_scans, 500, 82) input")
        if np.any(scans.std(axis=1) == 0):
            raise ValueError(f"{dataset}/{state}: constant regional time series")
        all_states[dataset][state] = np.stack([
            dg.temporal_harmonise(dg.regress_global_signal(scan), dataset)
            for scan in scans
        ])

generate_covmats(all_states, data_path)


Inspect resulting dictionary


In [ ]:
example_covmats = load_covariance_dict(data_path + '/covariance_matrices_gc.h5')

# Inspect the newly generated primary matrices
print("Datasets in file:")
datasets = example_covmats.keys()
print(list(datasets))

# Print all states within MA dataset
print("\nStates in MA dataset:")
ma_states = example_covmats['MA'].keys()
print(list(ma_states))

# Print shape of deep_propofol covariance matrices
print("\nShape of deep_propofol covariance matrices:")
deep_propofol_shape = example_covmats['MA']['deep_propofol'].shape
print(f"  {deep_propofol_shape}  (n_subjects, 82_regions, 82_regions)")


Inspect the DBS states alongside MA in the regenerated primary file:


In [ ]:
all_covmats = load_covariance_dict(data_path + '/covariance_matrices_gc.h5')

# Print all datasets (should be 'MA')
print("Datasets in file:")
datasets = all_covmats.keys()
print(list(datasets))

print("\nStates in MA dataset:")
ma_states = all_covmats['MA'].keys()
print(list(ma_states))
print("\nStates in DBS dataset:")
dbs_states = all_covmats['DBS'].keys()
print(list(dbs_states))

print("\nShape of deep_propofol covariance matrices:")
deep_propofol_shape = all_covmats['MA']['deep_propofol'].shape
print(f"  {deep_propofol_shape}  (n_subjects, 82_regions, 82_regions)")
print("\nShape of ts_on_3V covariance matrices:")
ts_on_3V_shape = all_covmats['DBS']['ts_on_3V'].shape
print(f"  {ts_on_3V_shape}  (n_subjects, 82_regions, 82_regions)")
